# 🚀 Free Cloud GPU Fine-Tuning Lab with Unsloth & QLoRA
### Train LLaMA 3.1 8B or Mistral 7B on a 100% Free Google Colab / Kaggle T4 GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---
### 📌 What this notebook accomplishes:
1. **Verifies your Free Cloud GPU** (NVIDIA Tesla T4 15GB or Kaggle Dual-T4).
2. **Mounts Google Drive** for persistent checkpoint saving.
3. **Loads LLaMA 3.1 8B in 4-bit NF4** using Unsloth (under 4GB VRAM!).
4. **Applies LoRA Adapters** to all attention and MLP projections.
5. **Formats & Prepares Custom Datasets** (Alpaca/ShareGPT format).
6. **Fine-tunes the model** using `SFTTrainer` and `paged_adamw_8bit`.
7. **Tests live inference** with the trained model.
8. **Exports to GGUF (Q4_K_M)** for running locally on your PC with Ollama or LM Studio!

## 1. Environment & GPU Verification

In [ ]:
# Check NVIDIA GPU Status
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Please go to Runtime -> Change runtime type -> Select T4 GPU!")

## 2. Mount Google Drive (Persistent Storage to Avoid Data Loss)

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/unsloth_finetuned_model"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will be safely stored at: {SAVE_DIR}")

## 3. Install Unsloth, XFormers & Fine-Tuning Dependencies

In [ ]:
%%capture
# Install Unsloth with fast OpenAI Triton kernels
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes datasets

## 4. Load 4-Bit Pre-Quantized Base Model (Llama 3.1 8B)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Context window size
dtype = None           # Auto detect (Float16 for Tesla T4)
load_in_4bit = True    # 4-bit QLoRA enabled

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"✅ Model loaded! GPU VRAM Allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")

## 5. Inject LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA Rank (8 or 16 are ideal for consumer GPUs)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,                       # Optimized to 0 for Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth", # Saves massive VRAM
    random_state=3407,
    max_seq_length=max_seq_length,
)

print("✅ LoRA Adapters injected successfully!")

## 6. Prepare & Format Training Dataset (Alpaca Style)

In [ ]:
from datasets import Dataset

# Sample domain dataset (replace with your domain JSON file)
sample_dataset = [
    {
        "instruction": "What is Low-Rank Adaptation (LoRA)?",
        "input": "",
        "output": "LoRA freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture, reducing the number of trainable parameters by over 90%."
    },
    {
        "instruction": "Write a Python function to check if a word is a palindrome.",
        "input": "",
        "output": "def is_palindrome(word: str) -> bool:\n    cleaned = word.lower().replace(' ', '')\n    return cleaned == cleaned[::-1]"
    },
    {
        "instruction": "Explain why QLoRA saves memory during LLM fine-tuning.",
        "input": "",
        "output": "QLoRA quantizes the base model weights to a special 4-bit NormalFloat (NF4) data type and uses Double Quantization to compress quantization constants, allowing billion-scale models to fit into consumer GPU VRAM."
    }
]

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_list(sample_dataset)
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"✅ Processed {len(dataset)} training examples.")

## 7. Start Training with SFTTrainer & PagedAdamW8bit

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,                       # Adjust for full dataset training
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="paged_adamw_8bit",           # Memory-efficient optimizer
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
    ),
)

# Execute training
trainer_stats = trainer.train()
print("🎉 Training complete!")

## 8. Test Live Inference (Post-Training Evaluation)

In [ ]:
# Enable Fast Inference Mode
FastLanguageModel.for_inference(model)

test_prompt = alpaca_prompt.format(
    "Explain why QLoRA saves memory during LLM fine-tuning.", # Instruction
    "",                                                      # Input
    "",                                                      # Output (to be generated)
)

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

## 9. Save Checkpoint & Export to GGUF (for Ollama / LM Studio)

In [ ]:
# 1. Save LoRA adapter directly to Google Drive (~100MB)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"✅ LoRA weights saved to Google Drive: {SAVE_DIR}")

# 2. Convert to GGUF (4-bit Q4_K_M) for offline local execution in Ollama / LM Studio
# model.save_pretrained_gguf("finetuned_model_q4", tokenizer, quantization_method="q4_k_m")
# print("✅ GGUF exported! Download and run with: ollama run finetuned_model_q4")